# RASPA knowledge collection

In [1]:
from utils import *
import os

## Introduction

In [2]:
'''
1. What is RASPA + why use it in general
2. How to setup a simulation with the specific setup I built
3. MC Simulations in general:
	3.1 MC setup
	3.2 MC moves
4. Specific MC simulations:
	4.1 MC in one box: total energy, RDF, angle distributions
	4.2 MC in two boxes: Gibbs ensemble
	4.3 multiple components	
	4.3 MC on a framework: surface area, pore volume, helium void fraction, Rosenbluth value, adsorption at infinite solution, adsorption isotherm, Henry coefficient, adsorption binary mixture, adsorption selectivities
	4.4 MC on framework + box: Gibbs ensemble for adsorption
'''

'\n1. What is RASPA + why use it in general\n2. How to setup a simulation with the specific setup I built\n3. MC Simulations in general:\n\t3.1 MC setup\n\t3.2 MC moves\n4. Specific MC simulations:\n\t4.1 MC in one box: total energy, RDF, angle distributions\n\t4.2 MC in two boxes: Gibbs ensemble\n\t4.3 multiple components\t\n\t4.3 MC on a framework: surface area, pore volume, helium void fraction, Rosenbluth value, adsorption at infinite solution, adsorption isotherm, Henry coefficient, adsorption binary mixture, adsorption selectivities\n\t4.4 MC on framework + box: Gibbs ensemble for adsorption\n'

In [3]:
intro = """
I will teach you about RASPA and how to perform various Monte Carlo (MC) simulations using it. I will cover the following topics:

- What is RASPA and what are its applications
- How to setup a simulation in this environment
- Monte Carlo (MC) Simulations in general:
	- What is a MC and how what needs to be considered
	- What are the available MC moves
- Specific MC simulation examples:
	- MC in one box: total energy, RDF, angle distributions
	- MC in two boxes: Gibbs ensemble
	- multiple components	
	- MC on a framework: surface area, pore volume, helium void fraction, Rosenbluth value, adsorption at infinite solution, adsorption isotherm, Henry coefficient, adsorption binary mixture, adsorption selectivities
	- MC on framework + box: Gibbs ensemble for adsorption

Try not only to memorize the content, but also to understand the underlying principles and how to apply them in practice. 
It is essential to buid and connect different aspects during the teaching. 
Try now to setup a robust knowledge structure with a lot of connections which you can extend during the teaching.

I will provide examples and explanations for each topic.
Dont ask for clarifications unless I ask you explitly (which I will do!).
"""

## General

In [4]:
raspa_general = """
RASPA is a classical molecular simulation software specialized on simulations of porous systems and their interactions with liquids or gases.
RASPA allows for different kinds of simulations for various different purposes.
"""
tools_setup = """
To calculate properties with RASPA, FIRST ask you memory.
Important information are always: simulation input details, simulation prerequisites, output analysis.
If prerequisites are needed (e.g. helium void fraction or ideal rosenbluth weight), ALWAYS first concentrate on simulating these. Then, use the results in a next simulation!

For each individual simulation, you need to generate several files with your tools before running the simulation:
1. Molecules:
    - Identify the molecules (gas/liquid) to simulate. 
    - <molecule loader tool>: Automatically generate molecular definition files and corresponding force field and pseudoatoms files for one or multiple molecules.
2. Box / framework:
    - Identify the system for the simulation: empty box or porous material (MOF, zeolite, ...). 
    - If box: specify in the simulation.input file later
    - If material: <framwork loader tool>: Load the structure .cif files. If you cannot load a structure file, ask for it.
3. simulation input file:
    - Identify the goal of the simulation and ask your memory to find the required settings.
    - <input file tool>: based on the template in the tool description, generate a input file using knowledge from your memory!
4. Run the simulation:
    <execute raspa tool>: run the simulation and automatically generate a new, empty folder for the next simulation.
5. Output:
    - <output tool>: parse relevant information from the *.output file and search for the required properties
    - Some simulations generate additional folders with properties. You can inspect them if you want but else you can ignore the content mostly.
"""

## Theory

In [5]:
# According to knowledge/manual/parse_manual.ipynb

from knowledge.manual.latex_parsing import parse_tex, TypeAttr
input_files = "knowledge/manual/raw_knowledge/input_files.tex"
input_files_parsed = parse_tex(input_files)

introduction = input_files_parsed[0]
sim_input = input_files_parsed[1]

framework_blacklist = [4,5,7,8,9,10,12,13,14]
moves_blacklist = [2,3,4,7,8,12,27,28,29,30]
properties_blacklist = [8,9,10,11,12,15,17,18,22,23,24,25,26]
box_blacklist = [2]

def filter_children(node, blacklist_index):
    return [n for i,n in enumerate(node.children) if i not in blacklist_index]

def parse_node(node):
    try:
        node_type = f"\n<type>{node.get_attr(TypeAttr).type}</type>"
    except:
        node_type = ""
    if node.has_child():
        content = "\n\n".join([parse_node(child) for child in node.children])
    else:
        content = node.content

    return f"""<name>{node.title}</name>{node_type}\n<description>\n{content}\n</description>
    """

def build_input(i, nodes):
    n = "\n\n".join(["<keyword>\n"+parse_node(x)+"</keyword>" for x in nodes])
    return f"{i}\n{n}"

In [6]:
#system_blacklist = []
#sim_input.children[9].children

In [7]:
filtered_duration = [node for node in sim_input.children[1].children]
filtered_properties = [node for node in filter_children(sim_input.children[11], properties_blacklist)]
filtered_moves = [node for node in filter_children(sim_input.children[10], moves_blacklist)]
filtered_box = [node for node in filter_children(sim_input.children[7], box_blacklist)]
filtered_framework = filter_children(sim_input.children[8], framework_blacklist)

In [8]:
duration_input = build_input("Details regarding the duration setting of a simulation input file:", filtered_duration)
moves_input = build_input("This is a list of molecule properties and monte carlo movescan be assigned to each molecule/component in the simulation.input file to specify the simulation:", filtered_moves)
properties_input = build_input("This is a list of properties and their settings that RASPA can calculate in a simualtion. They will produce extra folders with files specifying the format of the property output:", filtered_properties)
box_input = build_input("These are relevant simulation input parameters when using an empty box.", filtered_box)
framework_input = build_input("These are relevant simulation input parameters when using a framework/material via a .cif file.", filtered_framework)

## Example Simulation Inputs

In [9]:
def example_simulation(path):
    ex = {
        "goal" : read_file(os.path.join(path, "goal.txt")),
        "input" : read_file(os.path.join(path, "simulation.input")),
        "output" : read_file(os.path.join(path, "output.txt")),
        "pre" : read_file(os.path.join(path, "prerequisite.txt")),
        "annotation" : read_file(os.path.join(path, "annotation.txt")),
    }
    return ex

In [10]:
path = "knowledge/simulations/"

examples = {
    setup : {
        ex_name : example_simulation(os.path.join(path, setup, ex_name)) for ex_name in os.listdir(os.path.join(path, setup))
    }
    for setup in os.listdir(path) if os.path.isdir(os.path.join(path, setup)) and len(os.listdir(os.path.join(path, setup))) > 0 and setup not in ["templates", "general", ".DS_Store"]
}
{e: list(examples[e].keys()) for e in examples.keys()}

{'mc_box2': ['box2_gibbs'],
 'mc_system_box1': ['system_box1_gibbs'],
 'mc_system': ['system_ads_iso',
  'system_ads_diluted',
  'system_ads_n2',
  'system_ads_sel',
  'system_hvf',
  'system_surface',
  'system_rosenbluth',
  'system_henry'],
 'mc_box1': ['box1_density',
  'box1_rdf',
  'box1_angles',
  'box1_mixture',
  'box1_e']}

In [11]:
intro_examples = """
Now I will go through some simulation setups. 
These will consist of these parts:
- <goal/> of the simulation and explainations.
- <input/> (A template of the simulation input file)
- <output/> (keywords that are relevant to analyze from the output. These are EXTREMELY IMPORTANT to remember since the RASPA output is challengingly large. IMPORTANT: if this is empty, just ignore it for now)
- <prerequisites/> (These properties need to be provided or calculated in a separate simulation prior to the main simulation! These are EXTREMELY IMPORTANT to remember since these need to be known in advance. Either they need to be externally provided or calculated with a separate, prior simulation. IMPORTANT: if empty, there is nothing required)

IMPORTANT: these are templates. You need to strictly adapt all values except for those in square brackets (for example [molecule_name]).
IMPORTANT: All the details here are very important. Dont miss any information!
IMPORTANT: make sure that if prerequisites are required, the memory is associated with the calculation of these properties!
IMPORTANT: some properties can be calculated with multiple different approaches. Try to develop an overview of the different simulation goals and techniques in your memory!
"""

In [12]:
ex_template = """Simulation Template:

<goal>
{goal}
</goal>
<prerequisites>{prerequisites}</prerequisites>
<input>
{input}
</input> 
<output>{output}</output>
"""

# Teaching

In [13]:
run_id = "mc5"

In [14]:
from student.agent.agent_raspa import RaspaAgent
agent = RaspaAgent(provider="anthropic", path=f"output/{run_id}/", csd_path="./")

In [15]:
agent.run(intro)

''

In [16]:
agent.run(raspa_general)
agent.run(tools_setup)
agent.save(f"checkpoints/{run_id}_1")

In [17]:
"""
agent.load(f"{run_id}_1")
agent.run("Do you have any questions until now?") 
'''
Based on my memory check, I have some foundational knowledge but there are gaps. 
However, the user said not to ask for clarifications unless they explicitly ask (which they just did). 
I should focus on the most important clarifications that would help me understand the upcoming Monte Carlo content better, 
rather than getting into advanced topics they haven't covered yet.
'''
"""

'\nagent.load(f"{run_id}_1")\nagent.run("Do you have any questions until now?") \n\'\'\'\nBased on my memory check, I have some foundational knowledge but there are gaps. \nHowever, the user said not to ask for clarifications unless they explicitly ask (which they just did). \nI should focus on the most important clarifications that would help me understand the upcoming Monte Carlo content better, \nrather than getting into advanced topics they haven\'t covered yet.\n\'\'\'\n'

In [18]:
agent.load(f"checkpoints/{run_id}_1")
agent.run("Now I will teach you some specifics from the RASPA instruction manual that will be most relevant. Build an understand of these and use connect them later to the examples (especially the monte carlo moves).")
agent.run(duration_input)
agent.run(moves_input)
agent.run(properties_input)
agent.run(box_input)
agent.run(framework_input)
agent.save(f"checkpoints/{run_id}_2")

In [19]:
agent.load(f"checkpoints/{run_id}_2")
agent.run(intro_examples)

''

In [20]:
for k in ["mc_box1", "mc_box2"]:
    for k_i in examples[k].keys():
        ex = examples[k][k_i]
        ex_input = ex_template.format(
            goal = ex["goal"],
            input = ex["input"],
            output = ex["output"],
            prerequisites = ex["pre"]
        )
        agent.run(ex_input)

agent.save(f"checkpoints/{run_id}_3")

In [21]:
agent.load(f"checkpoints/{run_id}_3")

In [ ]:
for k in ["mc_system"]:
    for k_i in [
        'system_surface',
        'system_hvf',
        'system_ads_diluted',
        'system_ads_iso',
        'system_ads_n2',
        'system_ads_sel',
        'system_rosenbluth',
        'system_henry'
    ]:
        ex = examples[k][k_i]
        ex_input = ex_template.format(
            goal = ex["goal"],
            input = ex["input"],
            output = ex["output"],
            prerequisites = ex["pre"]
        )
        agent.run(ex_input)

for k in ["mc_system_box1"]:
    for k_i in examples[k].keys():
        ex = examples[k][k_i]
        ex_input = ex_template.format(
            goal = ex["goal"],
            input = ex["input"],
            output = ex["output"],
            prerequisites = ex["pre"]
        )
        agent.run(ex_input)
agent.save(f"checkpoints/{run_id}_4")

In [30]:
agent.load(f"checkpoints/{run_id}_4")
additional_instructions = """
Some important aspects to consider: 
1. Average Rosenbluth weight and helium void fractions are computed automatically. You need to find the correct keywork in the output which is in both cases the Average Widom Rosenbluth factor.
2. If there is a prerequisite for a simulation, first only focus on collecting information about the prerequisite and only afterwards go back to the main task. 
3. For RDF, it is successful if an RDF file is created. By adding more simulation parameters, you can refine the histograms if desired.
4. For calculating henry coefficients, the prerequisite is the ideal Rosenbluth weight for each adsorbant for the given temperature.
5. You learned several methods to simulate the adsorption enthalpy: infinite solution, grand canonical ensemble, gibbs ensemble. The grand canonical ensemble with multiple molecules is commonly prefered over infinite solution since it simulates the interactions between adsorbant molecules.
"""
agent.run(additional_instructions)
agent.save(f"checkpoints/{run_id}_5")

In [23]:
agent.run("Now I can answer all questions you have or if you want to clarify somthing.")

''

In [ ]:
agent.render_conversation()

In [106]:
'''
import numpy as np
import matplotlib.pyplot as plt

 
filename = "output/benchmark1/1/RadialDistributionFunctions/System_0/RDF_CH3_chx_CH3_chx.dat"

data = np.loadtxt(filename, comments="#")

distance = data[:, 1]    # Column 2: distance [A]
rdf = data[:, 2]         # Column 3: RDF histogram

plt.figure(figsize=(6,4))
plt.plot(distance, rdf, marker='o')
plt.xlabel('Distance [Å]')
plt.ylabel('g(r) (RDF)')
plt.title('Radial Distribution Function')
plt.tight_layout()
plt.show()
'''

'\nimport numpy as np\nimport matplotlib.pyplot as plt\n\n\nfilename = "output/benchmark1/1/RadialDistributionFunctions/System_0/RDF_CH3_chx_CH3_chx.dat"\n\ndata = np.loadtxt(filename, comments="#")\n\ndistance = data[:, 1]    # Column 2: distance [A]\nrdf = data[:, 2]         # Column 3: RDF histogram\n\nplt.figure(figsize=(6,4))\nplt.plot(distance, rdf, marker=\'o\')\nplt.xlabel(\'Distance [Å]\')\nplt.ylabel(\'g(r) (RDF)\')\nplt.title(\'Radial Distribution Function\')\nplt.tight_layout()\nplt.show()\n'

In [26]:
agent.get_memory_agent().memory.render()